# E-commerce Order Pipeline — Business Insights

This notebook produces four business-facing outputs from the gold layer.
Each chart answers a specific question a stakeholder would ask.
All data is loaded from pre-built gold CSVs — no database connection required.

---

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

GOLD_DIR = Path("../data/gold")
CHARTS_DIR = Path("charts")
CHARTS_DIR.mkdir(exist_ok=True)

# Clean, consistent chart style across all four outputs
plt.rcParams.update({
    "figure.figsize":      (11, 5),
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "axes.grid":           True,
    "grid.alpha":          0.25,
    "grid.linestyle":      "--",
    "font.size":           11,
    "axes.titlesize":      13,
    "axes.titleweight":    "bold",
    "axes.labelsize":      11,
    "xtick.labelsize":     10,
    "ytick.labelsize":     10,
})

PALETTE = {
    "North": "#2563EB",
    "South": "#16A34A",
    "East":  "#DC2626",
    "West":  "#D97706",
}

print("Setup complete. Gold CSVs ready to load.")

---
## Chart 1 — Revenue by Region Over Time

**Business question:** Which regions are growing and which are flat?

This chart shows monthly revenue per region over the full two-year window.
It helps stakeholders identify regional trends, seasonal patterns,
and whether investment in any region is paying off.


In [ ]:
rev = pd.read_csv(GOLD_DIR / "gold_revenue_by_region.csv", parse_dates=["order_month"])
rev = rev.sort_values("order_month")

fig, ax = plt.subplots()

for region, grp in rev.groupby("region"):
    ax.plot(
        grp["order_month"],
        grp["total_revenue"],
        label=region,
        color=PALETTE[region],
        linewidth=2.2,
        marker="o",
        markersize=3.5,
    )

ax.set_title("Monthly Revenue by Region")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (AUD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(title="Region", framealpha=0.4)
fig.tight_layout()
fig.savefig(CHARTS_DIR / "revenue_by_region.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: charts/revenue_by_region.png")

**Insight:** West consistently leads revenue across the period,
with all four regions showing a broadly upward trend.
North shows the flattest growth — a potential flag for regional strategy review.


---
## Chart 2 — Return Rate by Product Category

**Business question:** Which categories have a quality or expectation problem?

A high return rate signals either a product quality issue or a mismatch
between customer expectations and what they receive.
Clothing and Electronics tend to be the highest-return categories in real e-commerce,
and this synthetic dataset reflects that pattern.


In [ ]:
ret = pd.read_csv(GOLD_DIR / "gold_return_rate_by_category.csv")
ret = ret.sort_values("return_rate_pct", ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))

bars = ax.barh(
    ret["category"],
    ret["return_rate_pct"],
    color=["#93C5FD", "#60A5FA", "#3B82F6", "#1D4ED8"],
    edgecolor="white",
    height=0.55,
)

# Annotate each bar with the percentage value
for bar, val in zip(bars, ret["return_rate_pct"]):
    ax.text(
        bar.get_width() + 0.05,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.1f}%",
        va="center",
        fontsize=10,
        color="#374151",
    )

ax.set_title("Return Rate by Product Category")
ax.set_xlabel("Return Rate (%)")
ax.set_xlim(0, ret["return_rate_pct"].max() + 1.5)
ax.grid(axis="y", alpha=0)
fig.tight_layout()
fig.savefig(CHARTS_DIR / "return_rate_by_category.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: charts/return_rate_by_category.png")

**Insight:** Clothing leads return rate at ~6.1%, likely driven by sizing and fit issues.
Beauty has the lowest return rate — consistent with real-world patterns where
consumable or personal care products are rarely returned.


---
## Chart 3 — Top 10 Products by Revenue with Return Rate Overlay

**Business question:** Are our best-selling products also our most returned?

High revenue does not always mean high quality.
This chart surfaces the tension between revenue performance and return rate,
helping the business decide where to investigate product quality or listing accuracy.


In [ ]:
top = pd.read_csv(GOLD_DIR / "gold_top_products.csv")

# Truncate long product names for readability
top["short_name"] = top["product_name"].str[:28] + "..."

fig, ax1 = plt.subplots(figsize=(12, 5))

# Bar chart: revenue
bar_colors = [
    "#DC2626" if r > 8 else "#2563EB"
    for r in top["return_rate_pct"]
]
bars = ax1.bar(
    range(len(top)),
    top["total_revenue"],
    color=bar_colors,
    alpha=0.85,
    edgecolor="white",
    width=0.6,
)

ax1.set_ylabel("Total Revenue (AUD)")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax1.set_xticks(range(len(top)))
ax1.set_xticklabels(top["short_name"], rotation=35, ha="right", fontsize=9)
ax1.set_title("Top 10 Products by Revenue  |  Red = Return Rate > 8%")

# Return rate as text annotation on each bar
for i, (rev_val, ret_val) in enumerate(zip(top["total_revenue"], top["return_rate_pct"])):
    ax1.text(
        i, rev_val + 3000,
        f"{ret_val:.1f}%",
        ha="center", va="bottom",
        fontsize=8.5, color="#374151",
        fontweight="bold",
    )

fig.tight_layout()
fig.savefig(CHARTS_DIR / "top_products.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: charts/top_products.png")

**Insight:** The top revenue product also carries a ~9.6% return rate —
well above the category average of ~5.8% for Electronics.
This warrants a product listing or quality review before further marketing investment.
Blue bars indicate acceptable return rates; red bars flag products needing attention.


---
## Chart 4 — Customer Lifetime Value Segments

**Business question:** Where is our customer value concentrated?

Customers are segmented into High, Mid, and Low LTV tiers using tertiles
of lifetime revenue. This view helps prioritise retention spend —
keeping a High LTV customer is almost always cheaper than acquiring a new one.


In [ ]:
seg = pd.read_csv(GOLD_DIR / "gold_customer_segments.csv")

summary = (
    seg.groupby("ltv_segment")
    .agg(
        customer_count=("customer_id", "count"),
        avg_ltv=("lifetime_value", "mean"),
        avg_aov=("avg_order_value", "mean"),
    )
    .reindex(["High", "Mid", "Low"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 5))

seg_colors = {"High": "#16A34A", "Mid": "#2563EB", "Low": "#9CA3AF"}
bars = ax.bar(
    summary["ltv_segment"],
    summary["customer_count"],
    color=[seg_colors[s] for s in summary["ltv_segment"]],
    edgecolor="white",
    width=0.5,
)

# Annotate with avg LTV and avg order value
for bar, row in zip(bars, summary.itertuples()):
    label = "
".join([
        "Avg LTV: ${:,.0f}".format(row.avg_ltv),
        "Avg Order: ${:,.0f}".format(row.avg_aov),
    ])
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 4,
        label,
        ha="center", va="bottom",
        fontsize=9, color="#1F2937",
        linespacing=1.6,
    )

ax.set_title("Customer Count by LTV Segment")
ax.set_xlabel("LTV Segment")
ax.set_ylabel("Number of Customers")
ax.set_ylim(0, summary["customer_count"].max() + 60)
ax.grid(axis="x", alpha=0)
fig.tight_layout()
fig.savefig(CHARTS_DIR / "customer_segments.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: charts/customer_segments.png")

**Insight:** The three segments are roughly equal in customer count (~314 each),
but the High segment generates 3x the lifetime value of the Low segment ($24k vs $8k).
Retention spend should be heavily weighted toward the High and upper-Mid tiers.


---
## Summary

| Chart | Business Question | Key Finding |
|---|---|---|
| Revenue by Region | Which regions are growing? | West leads; North is flattest |
| Return Rate by Category | Where is the quality problem? | Clothing highest at ~6.1% |
| Top Products | Are top sellers also top returns? | #1 product has 9.6% return rate |
| Customer Segments | Where is value concentrated? | High LTV = 3x value of Low LTV |

All outputs saved to `notebooks/charts/` for use in the README.
